In [1]:
#%pip install jupysql duckdb duckdb-engine (can delete after run)

In [2]:
#1--turning on sql cells
#2--connect duckDB (mini database in notebook)
#3--results show as pandas database

In [3]:
%load_ext sql
%sql duckdb:// 
%config SqlMagic.autopandas = True

Connecting to 'duckdb://'

## Load the data

In [4]:
%%sql
--make a new table called hosp will overwrite if exist
--take all the columns
--reading in dataset
CREATE OR REPLACE TABLE hosp AS
SELECT *
FROM read_csv('../data/raw/Hospital_General_Information.csv', all_varchar = true);

Running query in 'duckdb://'

,Success


In [5]:
%%sql
--make a new table called tec will overwrite if exist
--take all the columns
--reading in dataset
    
CREATE OR REPLACE TABLE tec AS
SELECT *
FROM read_csv('../data/raw/Timely_and_Effective_Care_Hospital.csv', all_varchar = true);

Running query in 'duckdb://'

,Success


In [6]:
%%sql
-- 'hosp' AS table_name  -> a text label so we know which count is which
-- COUNT(*) AS n_rows             -> count every row and call that column n_rows
-- UNION ALL   -> stack the second result under the first

SELECT 'hosp' AS table_name, Count(*) AS n_rows FROM hosp
UNION ALL
SELECT 'tec', COUNT(*) FROM tec;

Running query in 'duckdb://'

,table_name,n_rows
0,hosp,5419
1,tec,138084


Hospital General information (hosp) has 5,419 rows
and Timely and Effective care dataset (tec) has 138,084 rows

In [7]:
%%sql
-- GOAL: count how many columns each table has
-- information_schema.columns has one row per column in every table,
-- counting rows per table_name = counting columns

SELECT table_name, COUNT(*) AS n_columns
FROM information_schema.columns
WHERE table_name IN ('hosp', 'tec')
GROUP BY table_name;

Running query in 'duckdb://'

,table_name,n_columns
0,tec,16
1,hosp,38


In [8]:
%%sql
SELECT *
FROM hosp
LIMIT 5;

Running query in 'duckdb://'

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,None,15,15,None,10,10,None
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,None,15,15,None,10,10,None
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,None,15,15,None,10,9,None
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,None,15,5,None,10,7,None
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29,15,10,29,10,7,29


In [9]:
%%sql
SELECT *
FROM tec
LIMIT 5;

Running query in 'duckdb://'

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Emergency Department,EDV,Emergency department volume,very high,None,None,01/01/2024,12/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,5,01/01/2024,12/31/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Not Available,Not Available,5,01/01/2024,12/31/2024


## Build the dataset

In [10]:
%%sql
--make a table of mental health ER wait times (OP_18c)
--change score and sample from text to numbers
CREATE OR REPLACE TABLE psych AS
SELECT "Facility ID" AS facility_id,
       TRY_CAST(Score AS INTEGER)  AS wait_psych,
       TRY_CAST(Sample AS INTEGER) AS n_psych_patients
FROM tec
WHERE "Measure ID" = 'OP_18c';

Running query in 'duckdb://'

,Success


In [11]:
%%sql
--make a table of regular ER wait times (OP_18b)
--change score and sample from text to numbers
CREATE OR REPLACE TABLE regular AS
SELECT "Facility ID" AS facility_id,
       TRY_CAST(Score AS INTEGER)  AS wait_regular,
       TRY_CAST(Sample AS INTEGER) AS n_regular_patients
FROM tec
WHERE "Measure ID" = 'OP_18b';

Running query in 'duckdb://'

,Success


In [12]:
%%sql
--make a table of ER volume (EDV)
--Score is a word here (low, medium, high, very high), so no TRY_CAST
CREATE OR REPLACE TABLE volume AS
SELECT "Facility ID" AS facility_id,
       Score AS er_volume
FROM tec
WHERE "Measure ID" = 'EDV';

Running query in 'duckdb://'

,Success


In [13]:
%%sql
--match psych, regular, and volume by hospital ID so all three are on one row
CREATE OR REPLACE TABLE waits AS
SELECT psych.facility_id,
       psych.wait_psych,
       psych.n_psych_patients,
       regular.wait_regular,
       regular.n_regular_patients,
       volume.er_volume
FROM psych
JOIN regular ON psych.facility_id = regular.facility_id
JOIN volume  ON psych.facility_id = volume.facility_id;

Running query in 'duckdb://'

,Success


In [14]:
%%sql
--add all hospital info (hosp) to the waits
--waits.* = all columns from waits, hosp.* = all 38 columns from hosp
CREATE OR REPLACE TABLE data AS
SELECT waits.*,
       hosp.*
FROM waits
JOIN hosp ON waits.facility_id = hosp."Facility ID";

Running query in 'duckdb://'

,Success


In [15]:
%%sql
--count the rows in the final table (should be 4,658)
SELECT COUNT(*) AS n_hospitals
FROM data;

Running query in 'duckdb://'

,n_hospitals
0,4658


In [16]:
%%sql
--look at the first 5 rows of the final table
SELECT *
FROM data
LIMIT 5;

Running query in 'duckdb://'

,facility_id,wait_psych,n_psych_patients,wait_regular,n_regular_patients,er_volume,Facility ID,Facility Name,Address,City/Town,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,<NA>,<NA>,212,396,very high,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,...,1,9,1,None,15,15,None,10,10,None
1,010005,242,30,143,1155,very high,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,...,1,8,0,None,15,15,None,10,10,None
2,010006,218,13,147,328,high,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,...,1,8,0,None,15,15,None,10,9,None
3,010007,242,57,130,1004,low,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,...,0,3,2,None,15,5,None,10,7,None
4,010011,<NA>,<NA>,158,341,Not Available,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,...,1,5,2,29,15,10,29,10,7,29


In [17]:
%%sql
--list every column in the final table and its type
--(DESCRIBE only says "Success" here, so we use information_schema)
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'data';

Running query in 'duckdb://'

,column_name,data_type
0,facility_id,VARCHAR
1,wait_psych,INTEGER
2,n_psych_patients,INTEGER
3,wait_regular,INTEGER
4,n_regular_patients,INTEGER
5,er_volume,VARCHAR
6,Facility ID,VARCHAR
7,Facility Name,VARCHAR
8,Address,VARCHAR
9,City/Town,VARCHAR


In [18]:
%%sql
--save the final table as a CSV in the Processed folder
COPY data TO '../data/Processed/boarding_dataset.csv' (HEADER, DELIMITER ',');

Running query in 'duckdb://'

,Success


In [19]:
%%sql
--check the saved CSV: did every hospital make it in?
--compare the CSV to the original tec counts (all four should match)
SELECT
    (SELECT COUNT(*) FROM tec WHERE "Measure ID" = 'OP_18c') AS psych_hospitals_in_tec,
    (SELECT COUNT(*) FROM tec WHERE "Measure ID" = 'OP_18b') AS regular_hospitals_in_tec,
    (SELECT COUNT(*) FROM tec WHERE "Measure ID" = 'EDV') AS volume_hospitals_in_tec,
    (SELECT COUNT(*) FROM read_csv('../data/Processed/boarding_dataset.csv')) AS hospitals_in_new_csv;

Running query in 'duckdb://'

,psych_hospitals_in_tec,regular_hospitals_in_tec,volume_hospitals_in_tec,hospitals_in_new_csv
0,4658,4658,4658,4658


The final dataset (boarding_dataset.csv) has 4,658 hospitals: both ER wait times, the number of patients for each, ER volume, plus all hospital info.